In [52]:
from dotenv import load_dotenv
load_dotenv()

True

In [53]:
import os
groq_api_key = os.getenv("GROQ_API_KEY") or None


In [54]:
from langchain_groq.chat_models import ChatGroq
model=ChatGroq(api_key=groq_api_key,model="openai/gpt-oss-120b") # type: ignore

In [55]:
from sentence_transformers import SentenceTransformer
embedder = SentenceTransformer('all-MiniLM-L6-v2')

Loading weights: 100%|██████████| 103/103 [00:00<00:00, 295.62it/s, Materializing param=pooler.dense.weight]                             
BertModel LOAD REPORT from: sentence-transformers/all-MiniLM-L6-v2
Key                     | Status     |  | 
------------------------+------------+--+-
embeddings.position_ids | UNEXPECTED |  | 

Notes:
- UNEXPECTED	:can be ignored when loading from different task/architecture; not ok if you expect identical arch.


In [56]:
from langchain_community.document_loaders import PyMuPDFLoader
from langchain_text_splitters import RecursiveCharacterTextSplitter
from langchain_community.vectorstores import FAISS
from langchain_core.embeddings import Embeddings


In [57]:
class Embeddings(Embeddings):
    def __init__(self, embedder: SentenceTransformer):
        self.embedder = embedder

    def embed_documents(self, texts):
        return self.embedder.encode(texts).tolist()

    def embed_query(self, text):
        return self.embedder.encode(text).tolist()
    
    def __call__(self, text):
        return self.embed_query(text)

In [94]:
class Rag:
    def __init__(self,model_name:ChatGroq,embedder:SentenceTransformer): 
        self.model=model_name
        self.embedding=Embeddings(embedder)
        self.text_splitter = RecursiveCharacterTextSplitter(chunk_size=100, chunk_overlap=50)
        self.vector_store=None
        self.path=None
    
    def create_vector_store(self,path:str):
        loader = PyMuPDFLoader(path)
        documents = loader.load()
        docs = self.text_splitter.split_documents(documents)
        self.vector_store = FAISS.from_documents(docs, self.embedding) #type:ignore
        return self.vector_store

    
    def retrieve(self,query:str):
        if self.vector_store is None:
            raise ValueError("Vector store not created.")
        relevant_docs = self.vector_store.similarity_search(query, k=10)  # Example: retrieve top 5 relevant documents
        return relevant_docs
    
    


    
    


In [95]:
path=r"C:\Users\user\Desktop\Hr\resume (22).pdf"

obj=Rag(model,embedder)

vector_store=obj.create_vector_store(path)

In [96]:
retrved_docs=obj.retrieve("Explain me the project of stock price prediction of your resume")

In [99]:
context = "\n\n".join([doc.page_content for doc in retrved_docs]).strip()

In [100]:
print(context)

PROJECTS
Stock Price & Sentiment Prediction Dashboard
GitHub

• Integrated 25 years of stock data and 5,000+ financial news sentiments using XGBoost to deliver

prediction models using Logistic Regression, Decision Trees,

• Developed and deployed ML models (MACD, RSI, regression) with Joblib, achieving an overall model

insights and identify investment opportunities through data-driven analytics.

reporting.
Loan Prediction Model
GitHub

Machine Learning:Regression, Classification, Clustering, Feature Engineering, Model Deployment

• Developed and compared multiple loan approval prediction models using Logistic Regression,

with Joblib, achieving an overall model accu-

news sentiments using XGBoost to deliver predictive
